# LSTMs for Text Classification

**Dataset:** AG_NEWS (News topic classification: World, Sports, Business, Sci/Tech)

**Instructions:** Complete the simple `# TODO` sections marked with `None`. Run all cells top-to-bottom to train and evaluate your model.

In [1]:
# Run this cell to install dependencies and load the dataset
!pip install -q datasets

In [2]:
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


### Step 1: Vocabulary & Preprocessing
Neural networks need numbers, not raw text. We will build a vocabulary to map words to integers.

In [ ]:
import re
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset

def tokenizer(text):
    return re.findall(r"[a-z0-9]+", text.lower())

ag_news = load_dataset('fancyzhx/ag_news')
train_data_shuffled = ag_news['train'].shuffle(seed=42).select(range(5000))
test_data_shuffled = ag_news['test'].shuffle(seed=42).select(range(1000))

# Using a small subset of data for fast training
train_list = [train_data_shuffled[i] for i in range(5000)]
test_list  = [test_data_shuffled[i] for i in range(1000)]

counter = Counter()
for example in train_list:
    counter.update(tokenizer(example['text']))

itos = ['<unk>', '<pad>'] + [w for w, c in counter.items() if c >= 2]
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX = stoi['<unk>']
PAD_IDX = stoi['<pad>']

def numericalize(text):
    return [stoi.get(tok, UNK_IDX) for tok in tokenizer(text)]

print(f"Vocabulary size: {len(itos):,}")

def collate_batch(batch):
    label_list, text_list = [], []
    for example in batch:
        label_list.append(example['label'])  # ag_news labels are already 0-indexed (0-3)
        processed_text = torch.tensor(numericalize(example['text']), dtype=torch.int64)
        text_list.append(processed_text)

    # Use nn.utils.rnn.pad_sequence to pad text_list so all sentences in the batch are the same length.
    padded_texts = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(label_list, dtype=torch.int64)

    return padded_texts, labels

tr_ld = DataLoader(train_list, batch_size=32, shuffle=True, collate_fn=collate_batch)
te_ld = DataLoader(test_list, batch_size=32, shuffle=False, collate_fn=collate_batch)


Vocabulary size: 10,505


### Step 2: Build the LSTM Model
Construct a simple LSTM text classifier.

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Define a PyTorch nn.LSTM layer (set batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Define a fully connected layer mapping hidden_dim to num_classes
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)

        # Pass the embedded text through the LSTM
        # The LSTM returns two things: output and (hidden_state, cell_state)
        output, (hidden, cell) = self.lstm(embedded)

        # We only care about the final hidden state of the last layer for classification
        final_hidden = hidden[-1]

        # Pass the final hidden state through the fully connected layer
        logits = self.fc(final_hidden)
        return logits

model = SimpleLSTM(vocab_size=len(itos), embed_dim=64, hidden_dim=128, num_classes=4).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Model parameters: 772,164


### Step 3: Train and Evaluate
Write the core steps of the PyTorch training loop.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for texts, labels in tr_ld:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        # Perform a forward pass
        predictions = model(texts)

        # Compute the loss
        loss = criterion(predictions, labels)

        # Perform backpropagation
        loss.backward()

        # Update the weights
        optimizer.step()

        total_loss += loss.item()
        correct += (predictions.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # Evaluation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for texts, labels in te_ld:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts)
            val_correct += (preds.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/3 | Train Acc: {train_acc:.1%} | Val Acc: {val_acc:.1%}")


Epoch 1/3 | Train Acc: 66.8% | Val Acc: 80.7%
Epoch 2/3 | Train Acc: 88.0% | Val Acc: 84.1%
Epoch 3/3 | Train Acc: 94.6% | Val Acc: 84.4%


### Step 4: Reflection

1. **What does the `padding_idx` argument do in the `nn.Embedding` layer?**
   - The `padding_idx` argument specifies a token index (here, `PAD_IDX` for `<pad>`) whose embedding vector is initialized to zeros and remains zero throughout training. Furthermore, gradients with respect to the `padding_idx` embedding vector are zeroed out during backpropagation, ensuring padding tokens do not affect parameter updates or introduce noise.

2. **Why do we extract `hidden[-1]` instead of using the raw `output` from the LSTM for classification?**
   - The raw `output` from the LSTM contains hidden state vectors for every individual time step across the sequence (shape: `[batch_size, seq_len, hidden_dim]`). In contrast, `hidden[-1]` extracts the final hidden state of the top LSTM layer after processing the entire sequence (shape: `[batch_size, hidden_dim]`). Because the LSTM processes tokens sequentially, `hidden[-1]` acts as a fixed-length summary encoding the full sentence context suitable for feeding into the linear classification layer.


### Step 5: Inference
Now that the model is trained, let's use it to classify a brand new headline that it has never seen before.

In [ ]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict(text, model):
    model.eval()

    # Convert the raw text into a tensor of token ids using numericalize().
    text_tensor = torch.tensor(numericalize(text), dtype=torch.int64).to(device)

    # The model expects a batch dimension. Add one with .unsqueeze(0).
    text_tensor = text_tensor.unsqueeze(0)

    with torch.no_grad():
        # Run a forward pass through the model to get the logits.
        logits = model(text_tensor)

        # Get the predicted class index from the logits (highest score).
        predicted_idx = logits.argmax(1).item()

    return class_names[predicted_idx]


sample_headlines = [
    "Manchester United wins dramatic final in extra time",
    "Central bank raises interest rates to combat inflation",
    "NASA's new telescope captures images of distant galaxy",
    "Peace talks resume between the two neighboring countries"
]

for headline in sample_headlines:
    prediction = predict(headline, model)
    print(f"'{headline}' -> {prediction}")


'Manchester United wins dramatic final in extra time' -> Sports
'Central bank raises interest rates to combat inflation' -> Business
'NASA's new telescope captures images of distant galaxy' -> Sci/Tech
'Peace talks resume between the two neighboring countries' -> World
